In [18]:
# =============================================================================
# Zelle 01 – Google Drive mounten
# =============================================================================
# Drive wird eingebunden, damit alle Dateien persistent gespeichert werden.
# Ohne Mount gehen alle Daten bei Session-Ende verloren.

from google.colab import drive

drive.mount('/content/drive')

# Basispfad definieren – EINMAL hier setzen, überall referenzieren
BASE_PATH = '/content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10'

print(f"Base Path: {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base Path: /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10


In [19]:
# =============================================================================
# Zelle 02 – Ordnerstruktur anlegen
# =============================================================================
# Alle benötigten Ordner werden einmalig erstellt.
# exist_ok=True verhindert Fehler wenn Ordner bereits existiert.

import os

# Alle Pfade relativ zu BASE_PATH definieren
folders = [
    'notebooks',
    'models',
    'reports/figures',
    'reports/metrics',
    'reports/misclassified',
    'presentation',
]

for folder in folders:
    full_path = os.path.join(BASE_PATH, folder)
    os.makedirs(full_path, exist_ok=True)
    print(f"✓ {full_path}")

print("\nOrdnerstruktur erfolgreich angelegt.")

✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/notebooks
✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/models
✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/reports/figures
✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/reports/metrics
✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/reports/misclassified
✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/presentation

Ordnerstruktur erfolgreich angelegt.


In [20]:
# =============================================================================
# Zelle 03 – GPU prüfen und dokumentieren
# =============================================================================
# Ohne GPU ist Training auf CIFAR-10 mit ResNet50 nicht praktikabel.
# Diese Zelle prüft welche Hardware verfügbar ist und dokumentiert dies.
# Bei "Not connected to a GPU" → Colab Runtime ändern:
# Laufzeit → Laufzeittyp ändern → T4 GPU

import tensorflow as tf

print("=" * 50)
print("HARDWARE CHECK")
print("=" * 50)

# TensorFlow Version
print(f"\nTensorFlow Version : {tf.__version__}")

# GPU verfügbar?
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        print(f"GPU erkannt        : {gpu.name}")

    # GPU Details via nvidia-smi
    print("\nGPU Details:")
    os.system('nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv')
else:
    print("WARNUNG: Keine GPU erkannt.")
    print("→ Laufzeit → Laufzeittyp ändern → T4 GPU auswählen")

# RAM Check
import psutil
ram = psutil.virtual_memory()
print(f"\nRAM total          : {ram.total / 1e9:.1f} GB")
print(f"RAM verfügbar      : {ram.available / 1e9:.1f} GB")

print("\n" + "=" * 50)

HARDWARE CHECK

TensorFlow Version : 2.20.0
GPU erkannt        : /physical_device:GPU:0

GPU Details:

RAM total          : 13.6 GB
RAM verfügbar      : 11.3 GB



In [21]:
# =============================================================================
# Zelle 04 – Zentrale Imports
# =============================================================================
# Alle Standard-Bibliotheken die im gesamten Notebook verwendet werden.
# Colab-spezifische und zellen-lokale Imports bleiben dort wo sie gebraucht werden.

# Standard Library
import os
import random
import warnings
warnings.filterwarnings('ignore')

# Numerik
import numpy as np

# Visualisierung
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras

# Reproduzierbarkeit – WICHTIG
# Alle Zufallsgeneratoren auf denselben Seed setzen damit Ergebnisse reproduzierbar sind
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Globale Konstanten
BASE_PATH = '/content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10'

print("Imports erfolgreich.")
print(f"NumPy     : {np.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"Seed      : {SEED}")

Imports erfolgreich.
NumPy     : 2.0.2
TensorFlow: 2.20.0
Seed      : 42


Warum SEED = 42 wichtig ist:
Ohne fixen Seed liefert jedes Training andere Ergebnisse. Kritische Analyse und Vergleich von Experimenten wird sonst unmöglich — Unterschiede könnten Zufall sein, nicht Methodenunterschied.

In [7]:
# =============================================================================
# Zelle 05 – GitHub Verbindung einrichten (DEAKTIVIERT)
# =============================================================================
# GRUND: Token war hardcoded im Code → Sicherheitsrisiko.
# LÖSUNG: Token wird ab Zelle 06 aus Colab Secrets geladen.
# Diese Zelle bleibt zur Dokumentation des Lernpfads erhalten.
# Token und E-Mail wurden unkenntlich gemacht.
#
# NICHT AUSFÜHREN
# =============================================================================
#
# import os
# from google.colab import userdata
#
# ── Konfiguration ──────────────────────────────────────────────────────────────
# GITHUB_USERNAME = "AwaTekoete"
# GITHUB_EMAIL    = "XXXX@XXXX.com"                  # ← ENTFERNT
# GITHUB_TOKEN    = "ghp_XXXXXXXXXXXXXXXXXXXX"        # ← ENTFERNT
# GITHUB_REPO_URL = f"https://{GITHUB_TOKEN}@github.com/AwaTekoete/MIST_CV_CIFAR10.git"
#
# REPO_LOCAL_PATH = '/content/MIST_CV_CIFAR10'
#
# ── Git Identität setzen ───────────────────────────────────────────────────────
# os.system(f'git config --global user.name "{GITHUB_USERNAME}"')
# os.system(f'git config --global user.email "{GITHUB_EMAIL}"')
#
# ── Repository klonen ─────────────────────────────────────────────────────────
# if not os.path.exists(REPO_LOCAL_PATH):
#     ret = os.system(f'git clone {GITHUB_REPO_URL} {REPO_LOCAL_PATH}')
#     if ret == 0:
#         print(f"✓ Repository geklont nach: {REPO_LOCAL_PATH}")
#     else:
#         print("✗ Klonen fehlgeschlagen – Token oder URL prüfen")
# else:
#     print(f"✓ Repository bereits vorhanden: {REPO_LOCAL_PATH}")
#
# ── Git Status – subprocess für korrekte Verzeichnis-Kontrolle ────────────────
# os.system() öffnet jedes Mal neue Shell → cd gilt nicht für nächsten Befehl
# subprocess mit cwd= setzt Arbeitsverzeichnis korrekt
# import subprocess
#
# result = subprocess.run(
#     ['git', 'status'],
#     cwd=REPO_LOCAL_PATH,          # ← Arbeitsverzeichnis explizit setzen
#     capture_output=True,
#     text=True
# )
#
# print("\nGit Status:")
# print(result.stdout)
#
# if result.returncode != 0:
#     print("Fehler:")
#     print(result.stderr)

✓ Repository bereits vorhanden: /content/MIST_CV_CIFAR10

Git Status:
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean



Sicherheitshinweis: Token niemals direkt in ein Notebook committen das auf GitHub landet. Nach dem Setup wird der Token aus dem Code entfernt und durch eine Umgebungsvariable ersetzt — das kommt in Zelle 06.

Zelle 06 — Token sichern (Sicherheit)
Token darf nie im Notebook-Code stehen der auf GitHub landet. Colab bietet dafür einen sicheren Secret-Speicher.
Einmalig in Colab einrichten:

Colab → linke Seitenleiste → Schlüsselsymbol (🔑 Secrets)
+ Add new secret
Name: GITHUB_TOKEN
Value: Token einfügen
Notebook access aktivieren

In [22]:
# =============================================================================
# Zelle 06 – Token-Sicherheit + Konfiguration zentralisieren
# =============================================================================
# Token wird aus Colab Secrets geladen – niemals hardcoded im Code.
# Diese Zelle muss bei jeder neuen Session ausgeführt werden.

import subprocess
from google.colab import userdata

# ── Konfiguration zentral ──────────────────────────────────────────────────────
CONFIG = {
    'base_path'    : '/content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10',
    'repo_path'    : '/content/MIST_CV_CIFAR10',
    'github_user'  : 'AwaTekoete',
    'github_email' : 'erik.gerst@hotmail.com',        # ← anpassen
    'seed'         : 42,
}

# ── Token aus Secrets laden ────────────────────────────────────────────────────
try:
    token = userdata.get('GITHUB_TOKEN')
    CONFIG['github_token'] = token

    # Remote URL mit Token aktualisieren (für Push)
    remote_url = f"https://{token}@github.com/AwaTekoete/MIST_CV_CIFAR10.git"
    subprocess.run(
        ['git', 'remote', 'set-url', 'origin', remote_url],
        cwd=CONFIG['repo_path'],
        capture_output=True
    )
    print("✓ Token aus Secrets geladen")
    print("✓ Remote URL aktualisiert")

except Exception as e:
    print(f"✗ Token nicht gefunden: {e}")
    print("→ Colab Secrets prüfen (🔑 linke Seitenleiste)")

# ── Konfiguration ausgeben (Token wird nicht angezeigt) ───────────────────────
print("\nAktive Konfiguration:")
for key, value in CONFIG.items():
    if key == 'github_token':
        print(f"  {key:<15}: ***")
    else:
        print(f"  {key:<15}: {value}")

✓ Token aus Secrets geladen
✓ Remote URL aktualisiert

Aktive Konfiguration:
  base_path      : /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10
  repo_path      : /content/MIST_CV_CIFAR10
  github_user    : AwaTekoete
  github_email   : erik.gerst@hotmail.com
  seed           : 42
  github_token   : ***


In [23]:
# =============================================================================
# Zelle 07 – GitHub Repository klonen
# =============================================================================
# Bereinigte Version von Zelle 05.
# Token wird aus CONFIG geladen – kein hardcoded Token im Code.
# VORAUSSETZUNG: Zelle 06 muss vor dieser Zelle ausgeführt sein.
# =============================================================================

import os
import subprocess

REPO_LOCAL_PATH = CONFIG['repo_path']
GITHUB_REPO_URL = f"https://{CONFIG['github_token']}@github.com/{CONFIG['github_user']}/MIST_CV_CIFAR10.git"

# ── Repository klonen (nur wenn noch nicht vorhanden) ─────────────────────────
if not os.path.exists(REPO_LOCAL_PATH):
    ret = os.system(f'git clone {GITHUB_REPO_URL} {REPO_LOCAL_PATH}')
    if ret == 0:
        print(f"✓ Repository geklont nach: {REPO_LOCAL_PATH}")
    else:
        print("✗ Klonen fehlgeschlagen – Token oder URL prüfen")
else:
    print(f"✓ Repository bereits vorhanden: {REPO_LOCAL_PATH}")

# ── Git Identität setzen ───────────────────────────────────────────────────────
os.system(f'git config --global user.name "{CONFIG["github_user"]}"')
os.system(f'git config --global user.email "{CONFIG["github_email"]}"')

# ── Git Status ────────────────────────────────────────────────────────────────
result = subprocess.run(
    ['git', 'status'],
    cwd=REPO_LOCAL_PATH,
    capture_output=True,
    text=True
)
print(f"\nGit Status:\n{result.stdout}")

if result.returncode != 0:
    print(f"✗ Fehler: {result.stderr}")

✓ Repository bereits vorhanden: /content/MIST_CV_CIFAR10

Git Status:
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	requirements.txt

nothing added to commit but untracked files present (use "git add" to track)



In [24]:
# =============================================================================
# Zelle 08 – Abhängigkeiten installieren + requirements.txt erstellen
# =============================================================================
# Alle benötigten Bibliotheken werden installiert und versioniert dokumentiert.
# requirements.txt ermöglicht reproduzierbare Umgebung auf jedem System.
# =============================================================================

import subprocess
import sys

# ── Bibliotheken installieren ─────────────────────────────────────────────────
packages = [
    'tensorflow',
    'numpy',
    'matplotlib',
    'seaborn',
    'scikit-learn',    # Metriken: F1, Confusion Matrix, Classification Report
    'Pillow',          # Bildverarbeitung
    'opencv-python',   # Bildqualitäts-Checks (Laplacian Variance etc.)
]

print("Installiere Pakete...")
for package in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', package, '-q'],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print(f"✓ {package}")
    else:
        print(f"✗ {package} – Fehler: {result.stderr[:100]}")

# ── Installierte Versionen erfassen ───────────────────────────────────────────
import tensorflow as tf
import numpy as np
import matplotlib
import seaborn as sns
import sklearn
import PIL
import cv2

versions = {
    'tensorflow'      : tf.__version__,
    'numpy'           : np.__version__,
    'matplotlib'      : matplotlib.__version__,
    'seaborn'         : sns.__version__,
    'scikit-learn'    : sklearn.__version__,
    'Pillow'          : PIL.__version__,
    'opencv-python'   : cv2.__version__,
}

print("\nInstallierte Versionen:")
for lib, ver in versions.items():
    print(f"  {lib:<20}: {ver}")

# ── requirements.txt erstellen ────────────────────────────────────────────────
req_path = os.path.join(CONFIG['base_path'], 'requirements.txt')

with open(req_path, 'w') as f:
    f.write("# MIST CV CIFAR-10 – Requirements\n")
    f.write("# Generiert automatisch – Zelle 08\n\n")
    for lib, ver in versions.items():
        f.write(f"{lib}=={ver}\n")

print(f"\n✓ requirements.txt erstellt: {req_path}")

Installiere Pakete...
✓ tensorflow
✓ numpy
✓ matplotlib
✓ seaborn
✓ scikit-learn
✓ Pillow
✓ opencv-python

Installierte Versionen:
  tensorflow          : 2.20.0
  numpy               : 2.0.2
  matplotlib          : 3.10.0
  seaborn             : 0.13.2
  scikit-learn        : 1.6.1
  Pillow              : 11.3.0
  opencv-python       : 4.13.0

✓ requirements.txt erstellt: /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/requirements.txt


In [25]:
# =============================================================================
# Zelle 09 – Smoke Test
# =============================================================================
# Prüft ob alle Komponenten korrekt eingerichtet sind.
# Smoke Test = minimaler Test ob System grundsätzlich funktioniert.
# Kein inhaltlicher Test – nur Infrastruktur.
# =============================================================================

import os
import subprocess
import tensorflow as tf

print("=" * 55)
print("SMOKE TEST – Setup Verifikation")
print("=" * 55)

tests = []

# ── Test 1: Google Drive ──────────────────────────────────────────────────────
drive_ok = os.path.exists(CONFIG['base_path'])
tests.append(('Google Drive gemountet', drive_ok))

# ── Test 2: Ordnerstruktur ────────────────────────────────────────────────────
expected_folders = [
    'notebooks',
    'models',
    'reports/figures',
    'reports/metrics',
    'reports/misclassified',
    'presentation',
]
folders_ok = all(
    os.path.exists(os.path.join(CONFIG['base_path'], f))
    for f in expected_folders
)
tests.append(('Ordnerstruktur vollständig', folders_ok))

# ── Test 3: GPU ───────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
gpu_ok = len(gpus) > 0
tests.append(('GPU verfügbar', gpu_ok))

# ── Test 4: Seed gesetzt ──────────────────────────────────────────────────────
seed_ok = CONFIG['seed'] == 42
tests.append(('Seed konfiguriert', seed_ok))

# ── Test 5: GitHub Verbindung ─────────────────────────────────────────────────
result = subprocess.run(
    ['git', 'status'],
    cwd=CONFIG['repo_path'],
    capture_output=True,
    text=True
)
github_ok = result.returncode == 0
tests.append(('GitHub Verbindung aktiv', github_ok))

# ── Test 6: requirements.txt vorhanden ───────────────────────────────────────
req_ok = os.path.exists(os.path.join(CONFIG['base_path'], 'requirements.txt'))
tests.append(('requirements.txt vorhanden', req_ok))

# ── Test 7: TensorFlow Basisoperation ────────────────────────────────────────
try:
    x = tf.constant([1.0, 2.0, 3.0])
    y = tf.reduce_mean(x)
    tf_ok = float(y.numpy()) == 2.0
except Exception:
    tf_ok = False
tests.append(('TensorFlow Basisoperation', tf_ok))

# ── Ergebnis ausgeben ─────────────────────────────────────────────────────────
print()
all_passed = True
for test_name, passed in tests:
    status = "✓" if passed else "✗"
    print(f"  {status}  {test_name}")
    if not passed:
        all_passed = False

print()
print("=" * 55)
if all_passed:
    print("ERGEBNIS: Alle Tests bestanden – Setup vollständig.")
else:
    print("ERGEBNIS: Fehler gefunden – Details oben prüfen.")
print("=" * 55)

SMOKE TEST – Setup Verifikation

  ✓  Google Drive gemountet
  ✓  Ordnerstruktur vollständig
  ✓  GPU verfügbar
  ✓  Seed konfiguriert
  ✓  GitHub Verbindung aktiv
  ✓  requirements.txt vorhanden
  ✓  TensorFlow Basisoperation

ERGEBNIS: Alle Tests bestanden – Setup vollständig.


In [26]:
# =============================================================================
# Zelle 10 – Ersten Commit auf GitHub pushen
# =============================================================================
# Notebook und requirements.txt werden ins GitHub Repository kopiert
# und als erster Commit gespeichert.
# Konvention: chore: für Setup- und Konfigurations-Commits
# =============================================================================

import shutil
import subprocess
import os

REPO_PATH = CONFIG['repo_path']

# ── Ordnerstruktur im Repo anlegen ────────────────────────────────────────────
os.makedirs(os.path.join(REPO_PATH, 'notebooks'), exist_ok=True)
os.makedirs(os.path.join(REPO_PATH, 'models'), exist_ok=True)
os.makedirs(os.path.join(REPO_PATH, 'reports', 'figures'), exist_ok=True)
os.makedirs(os.path.join(REPO_PATH, 'reports', 'metrics'), exist_ok=True)
os.makedirs(os.path.join(REPO_PATH, 'reports', 'misclassified'), exist_ok=True)
os.makedirs(os.path.join(REPO_PATH, 'presentation'), exist_ok=True)
print("✓ Ordnerstruktur im Repo angelegt")

# ── requirements.txt kopieren ─────────────────────────────────────────────────
src_req = os.path.join(CONFIG['base_path'], 'requirements.txt')
dst_req = os.path.join(REPO_PATH, 'requirements.txt')
shutil.copy(src_req, dst_req)
print("✓ requirements.txt kopiert")

# ── Notebook kopieren ───────────────────────────────

✓ Ordnerstruktur im Repo angelegt
✓ requirements.txt kopiert
